In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

25/02/24 14:12:02 WARN Utils: Your hostname, LAPTOP-56LON6FU resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/02/24 14:12:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/02/24 14:12:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/02/24 14:12:03 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/02/24 14:12:03 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/02/24 14:12:03 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


In [3]:
spark.version

'3.5.4'

In [4]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet

--2025-02-24 14:12:03--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 108.138.245.96, 108.138.245.16, 108.138.245.58, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|108.138.245.96|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 64346071 (61M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2024-10.parquet.1’

yellow_tripdata_202 100%[===================>]  61.36M  4.83MB/s    in 14s     

2025-02-24 14:12:18 (4.46 MB/s) - ‘yellow_tripdata_2024-10.parquet.1’ saved [64346071/64346071]



In [5]:
!ls -lh yellow_tripdata_2024-10.parquet

-rwxrwxrwx 1 kench kench 62M Dec 19 05:21 yellow_tripdata_2024-10.parquet


In [6]:
# schema = types.StructType([
#     types.StructField('hvfhs_license_num', types.StringType(), True),
#     types.StructField('dispatching_base_num', types.StringType(), True),
#     types.StructField('pickup_datetime', types.TimestampType(), True),
#     types.StructField('dropoff_datetime', types.TimestampType(), True),
#     types.StructField('PULocationID', types.IntegerType(), True),
#     types.StructField('DOLocationID', types.IntegerType(), True),
#     types.StructField('SR_Flag', types.StringType(), True)
# ])

In [41]:
df = spark.read \
    .option("header", "true") \
    .parquet('yellow_tripdata_2024-10.parquet')
#     .schema(schema) \
df = df.repartition(4)

df.write.parquet('data/pq/yellow/2024/10/',mode='overwrite')

In [42]:
df = spark.read.parquet('data/pq/yellow/2024/10/')

**Q3**: How many taxi trips were there on October 15?

In [43]:
from pyspark.sql import functions as F

In [44]:
df.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee']

In [45]:
df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2024-10-03 03:40:19|  2024-10-03 03:46:11|              1|          0.6|         1|                 N|          48|         161|           1|        6.5|  3.5|    0.5|       2.

In [47]:
df \
    .withColumn('pickup_date', F.to_date(df.tpep_pickup_datetime)) \
    .filter("pickup_date = '2024-10-15'") \
    .count()

128893

In [48]:
df.registerTempTable('yellow_2024_10')

In [50]:
spark.sql("""
SELECT
    COUNT(1)
FROM 
    yellow_2024_10
WHERE
    to_date(tpep_pickup_datetime) = '2024-10-15';
""").show()

+--------+
|count(1)|
+--------+
|  128893|
+--------+



**Q4**: Longest trip for each day

In [23]:
df.columns

['hvfhs_license_num',
 'dispatching_base_num',
 'pickup_datetime',
 'dropoff_datetime',
 'PULocationID',
 'DOLocationID',
 'SR_Flag']

In [70]:
df \
    .withColumn('duration', (F.unix_timestamp(df.tpep_dropoff_datetime) - F.unix_timestamp(df.tpep_pickup_datetime))/3600) \
    .withColumn('pickup_date', F.to_date(df.tpep_pickup_datetime)) \
    .groupBy('pickup_date') \
    .max('duration') \
    .orderBy('max(duration)', ascending=False) \
    .limit(5) \
    .show()

+-----------+------------------+
|pickup_date|     max(duration)|
+-----------+------------------+
| 2024-10-16|162.61777777777777|
| 2024-10-03|           143.325|
| 2024-10-22|137.76055555555556|
| 2024-10-18|114.83472222222223|
| 2024-10-21| 89.89833333333333|
+-----------+------------------+



In [77]:
spark.sql("""
SELECT
    to_date(tpep_pickup_datetime) AS pickup_date,
    MAX((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600) AS duration
FROM 
    yellow_2024_10
GROUP BY
    1
ORDER BY
    2 DESC
LIMIT 5;
""").show()

+-----------+------------------+
|pickup_date|          duration|
+-----------+------------------+
| 2024-10-16|162.61777777777777|
| 2024-10-03|           143.325|
| 2024-10-22|137.76055555555556|
| 2024-10-18|114.83472222222223|
| 2024-10-21| 89.89833333333333|
+-----------+------------------+



**Q5**: Most frequent `dispatching_base_num`

How many stages this spark job has?



In [79]:
# spark.sql("""
# SELECT
#     dispatching_base_num,
#     COUNT(1)
# FROM 
#     yellow_2024_10
# GROUP BY
#     1
# ORDER BY
#     2 DESC
# LIMIT 5;
# """).show()

In [28]:
# df \
#     .groupBy('dispatching_base_num') \
#         .count() \
#     .orderBy('count', ascending=False) \
#     .limit(5) \
#     .show()

+--------------------+-------+
|dispatching_base_num|  count|
+--------------------+-------+
|                NULL|3833771|
+--------------------+-------+



**Q6**: Most common locations pair

In [29]:
df_zones = spark.read.parquet('zones')

In [30]:
df_zones.columns

['LocationID', 'Borough', 'Zone', 'service_zone']

In [31]:
df.columns

['hvfhs_license_num',
 'dispatching_base_num',
 'pickup_datetime',
 'dropoff_datetime',
 'PULocationID',
 'DOLocationID',
 'SR_Flag']

In [32]:
df_zones.registerTempTable('zones')

In [83]:
spark.sql("""
SELECT
    CONCAT(pul.Zone, ' / ', dol.Zone) AS pu_do_pair,
    COUNT(1)
FROM 
    yellow_2024_10 fhv LEFT JOIN zones pul ON fhv.PULocationID = pul.LocationID
                      LEFT JOIN zones dol ON fhv.DOLocationID = dol.LocationID
GROUP BY 
    1
ORDER BY
    2 ASC
LIMIT 10;
""").show()

+--------------------+--------+
|          pu_do_pair|count(1)|
+--------------------+--------+
|Kew Gardens Hills...|       1|
|Mount Hope / Stuy...|       1|
|Cypress Hills / C...|       1|
|Central Harlem / ...|       1|
|Soundview/Bruckne...|       1|
|Inwood / Pelham P...|       1|
|Woodside / West F...|       1|
|Williamsbridge/Ol...|       1|
|Queensbridge/Rave...|       1|
|   Jamaica / Gowanus|       1|
+--------------------+--------+

